# KV260 DPU Benchmark Results
**Platform:** AMD Kria KV260 · DPU B512 · XRT 2.13.479  
**Run date:** 2026-08-12 · 10 rounds × 100 frames per model  
**Models:** ResNet50 · InceptionV1 · YOLOv3

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# 10-round averages from bench_run.log
data = {
    'Model':       ['ResNet50',  'ResNet50',  'InceptionV1', 'InceptionV1', 'YOLOv3',  'YOLOv3'],
    'Mode':        ['DPU',       'CPU',       'DPU',         'CPU',         'DPU',     'CPU'],
    'FPS':         [84.4,        1.58,        165.8,         3.86,          13.2,      0.22],
    'Latency(ms)': [11.9,        635.0,       6.0,           259.0,         75.6,      4646.0],
    'Power(W)':    [7.78,        6.01,        7.44,          5.99,          9.11,      6.12],
    'FPS/Watt':    [10.85,       0.26,        22.28,         0.64,          1.45,      0.04],
}
df = pd.DataFrame(data)

# Speedup table
dpu = df[df.Mode == 'DPU'].set_index('Model')
cpu = df[df.Mode == 'CPU'].set_index('Model')
speedup = (dpu['FPS'] / cpu['FPS']).round(0).astype(int)
eff_gain = (dpu['FPS/Watt'] / cpu['FPS/Watt']).round(0).astype(int)

summary = pd.DataFrame({
    'DPU FPS': dpu['FPS'],
    'CPU FPS': cpu['FPS'],
    'Speedup': speedup.astype(str) + 'x',
    'DPU Latency(ms)': dpu['Latency(ms)'],
    'DPU Power(W)': dpu['Power(W)'],
    'Efficiency Gain': eff_gain.astype(str) + 'x',
})
print('=== KV260 DPU vs ARM CPU — 10-Round Average ===')
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
models = ['ResNet50', 'InceptionV1', 'YOLOv3']
colors = ['#1f77b4', '#ff7f0e']

for ax, model in zip(axes, models):
    row = df[df.Model == model]
    fps_vals = row['FPS'].values
    bars = ax.bar(['DPU', 'CPU'], fps_vals, color=colors, width=0.5)
    ax.set_title(model, fontsize=13, fontweight='bold')
    ax.set_ylabel('FPS')
    for bar, val in zip(bars, fps_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f'{val:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    sp = int(fps_vals[0] / fps_vals[1])
    ax.set_xlabel(f'{sp}x faster on DPU', fontsize=10, color='green', fontweight='bold')

fig.suptitle('KV260 DPU vs ARM CPU — Throughput (FPS)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(models))
w = 0.35
dpu_eff = [10.85, 22.28, 1.45]
cpu_eff = [0.26,  0.64,  0.04]

b1 = ax.bar([i - w/2 for i in x], dpu_eff, w, label='DPU', color='#1f77b4')
b2 = ax.bar([i + w/2 for i in x], cpu_eff, w, label='CPU', color='#ff7f0e')
ax.set_xticks(list(x))
ax.set_xticklabels(models)
ax.set_ylabel('FPS / Watt')
ax.set_title('Energy Efficiency: FPS per Watt', fontsize=13, fontweight='bold')
ax.legend()
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}', ha='center', fontsize=10, fontweight='bold', color='#1f77b4')
plt.tight_layout()
plt.show()
print('\nKey finding: DPU delivers 17–42x better energy efficiency than ARM CPU')